# Export Model B predictions + physical features for all 514 Testset samples (issue #19)

`model_analysis_summary.json` only has Model B's `pred_raw` and its 4 physical
features (MFE, ΔG, Tm, GC) for the 15 Case Studies. The Model B explanation
dashboard (`visualization/plan_draft.md`) needs them for the whole Testset.

This notebook runs `export_model_b_testset.py` on GPU and writes
`final_analysis_result/model_b_testset_predictions.csv` (one row per
`sample_id`, 0-513). The script refuses to write the file unless its 15 Case
Study rows match `model_analysis_summary.json`. Integrated Gradients is not
computed here (Case Studies only).

## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Point at your uploaded assets

Edit `DRIVE_DIR` if needed. `final_analysis_result/model_analysis_summary.json`
is the committed file from your local `xai_test/` checkout - upload it with the
same folder layout.

In [ ]:
import os

DRIVE_DIR = "/content/drive/MyDrive/xai_test"  # <-- EDIT ME

required = [
    "NT_sacas9_fintuned_model",
    "hybrid_xgb_model.json",
    "test_metadata.csv",
    "final_analysis_result/model_analysis_summary.json",
]
missing = [
    name for name in required if not os.path.exists(os.path.join(DRIVE_DIR, name))
]
if missing:
    raise FileNotFoundError(
        f"Missing in {DRIVE_DIR}: {missing}. Upload these to Drive first, or fix DRIVE_DIR."
    )

os.chdir(DRIVE_DIR)
print("Working directory:", os.getcwd())

## 3. Install ViennaRNA CLI (RNAfold / RNAduplex) and Python deps

In [ ]:
!apt-get -qq update && apt-get -qq install -y vienna-rna
!which RNAfold && which RNAduplex

In [ ]:
!pip install -q transformers biopython xgboost joblib

## 4. Write the predictor + export modules

Mirrors the committed `xai_test/` repo exactly (embedded at notebook-build
time, not fetched at runtime).

In [ ]:
%%writefile model_b_wrapper.py
import os
import pickle
import re
import subprocess

import joblib
import numpy as np
import torch
import xgboost as xgb
from Bio.SeqUtils import MeltingTemp as mt
from Bio.SeqUtils import gc_fraction
from transformers import AutoModelForMaskedLM, AutoTokenizer


class Model_B_Predictor:
    def __init__(self, nt_model_dir: str, xgb_model_path: str):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        print(f"Loading NT backbone from {nt_model_dir}...")
        self.tokenizer = AutoTokenizer.from_pretrained(nt_model_dir)
        self.nt_model = AutoModelForMaskedLM.from_pretrained(nt_model_dir).to(
            self.device
        )
        self.nt_model.eval()

        print(f"Loading XGBoost model from {xgb_model_path}...")
        self.xgb_model = self._load_xgb_model(xgb_model_path)

    def _load_xgb_model(self, path: str):
        if not os.path.exists(path):
            raise FileNotFoundError(f"XGBoost model file not found: {path}")

        try:
            with open(path, "rb") as f:
                return pickle.load(f)
        except Exception:  # noqa: BLE001 - pre-existing multi-format load fallback
            try:
                return joblib.load(path)
            except Exception:  # noqa: BLE001 - pre-existing multi-format load fallback
                model = xgb.XGBRegressor()
                model.load_model(path)
                return model

    def extract_nt_embeddings(self, sequences: list[str]) -> np.ndarray:
        inputs = self.tokenizer(
            sequences, return_tensors="pt", padding=True, truncation=True
        ).to(self.device)
        with torch.no_grad():
            outputs = self.nt_model(**inputs, output_hidden_states=True)
            last_hidden_state = outputs.hidden_states[-1]

            batch_size, seq_len, hidden_dim = last_hidden_state.shape
            assert seq_len == 7, f"Expected 7 tokens, got {seq_len}"
            assert hidden_dim == 1280, f"Expected 1280 hidden dim, got {hidden_dim}"

            flattened_embeddings = last_hidden_state.view(batch_size, -1).cpu().numpy()
        return flattened_embeddings

    def _get_guide_rna(self, dna_seq: str) -> str:
        complement_map = str.maketrans("ATCGatcg", "UAGCuagc")
        return dna_seq.translate(complement_map)[::-1]

    # RNAfold/RNAduplex are external processes: each subprocess.run() call pays
    # OS process-spawn overhead regardless of how little work it does. Both CLIs
    # accept many records over a single stdin stream, so batching sequences into
    # one call (chunked, to keep any single call's stdin/stdout bounded) turns an
    # O(n) process-spawn cost into O(n / batch_size) - the dominant cost at
    # ISM-sweep scale (tens of thousands of mutants). Deliberate scope override of
    # issue #9's "RNAfold/RNAduplex call batching" Out-of-Scope note, requested
    # directly by the user; verified to reproduce the exact same per-sequence
    # energy values as the prior one-process-per-sequence implementation.
    PHYSICAL_FEATURE_BATCH_SIZE = 500

    @staticmethod
    def _chunked(items: list, size: int):
        for i in range(0, len(items), size):
            yield items[i : i + size]

    def _calc_mfe_batch(self, seqs: list[str]) -> list[float]:
        energies: list[float] = []
        for chunk in self._chunked(seqs, self.PHYSICAL_FEATURE_BATCH_SIZE):
            # RNAfold CLI 호출 (--noPS: 이미지 생성 방지)
            proc = subprocess.run(
                ["RNAfold", "--noPS"],
                input="".join(f"{seq}\n" for seq in chunk),
                capture_output=True,
                text=True,
                check=True,
            )
            # Each record prints "<sequence>\n<dot-bracket structure> (<energy>)\n".
            # The structure diagram's own parentheses never have a digit directly
            # inside them, so this regex matches exactly one energy per record,
            # in input order. 출력 예시: ... ( -4.20)
            matches = re.findall(r"\(\s*(-?\d+(?:\.\d+)?)\)", proc.stdout)
            assert len(matches) == len(chunk), (
                f"RNAfold batch output mismatch: expected {len(chunk)} results, got {len(matches)}"
            )
            energies.extend(float(m) for m in matches)
        return energies

    def _calc_duplex_dg_batch(self, pairs: list[tuple[str, str]]) -> list[float]:
        energies: list[float] = []
        for chunk in self._chunked(pairs, self.PHYSICAL_FEATURE_BATCH_SIZE):
            # RNAduplex CLI 호출
            proc = subprocess.run(
                ["RNAduplex"],
                input="".join(f"{guide}\n{target}\n" for guide, target in chunk),
                capture_output=True,
                text=True,
                check=True,
            )
            # 출력 예시: ... :  1,36 :   1,36 (-32.50)
            matches = re.findall(r"\(\s*(-?\d+(?:\.\d+)?)\)", proc.stdout)
            assert len(matches) == len(chunk), (
                f"RNAduplex batch output mismatch: expected {len(chunk)} results, got {len(matches)}"
            )
            energies.extend(float(m) for m in matches)
        return energies

    def compute_physical_features(self, sequences: list[str]) -> np.ndarray:
        # 1. MFE (RNAfold.exe), batched across all sequences
        mfe_values = self._calc_mfe_batch(sequences)

        # 2. Duplex ΔG (RNAduplex.exe), batched across all sequences
        guide_rnas = [self._get_guide_rna(seq) for seq in sequences]
        dg_values = self._calc_duplex_dg_batch(list(zip(guide_rnas, sequences)))

        features = []
        for seq, mfe, dg in zip(sequences, mfe_values, dg_values):
            # 3. Tm (must match training-time nn_table=DNA_NN4 in nt_feature_stacking.py)
            tm = mt.Tm_NN(seq, nn_table=mt.DNA_NN4)

            # 4. GC Content
            gc = gc_fraction(seq) * 100.0

            features.append([mfe, dg, tm, gc])
        return np.array(features, dtype=np.float32)

    def predict(self, sequences: list[str]) -> np.ndarray:
        embeddings = self.extract_nt_embeddings(sequences)
        physical_feats = self.compute_physical_features(sequences)

        final_features = np.hstack([embeddings, physical_feats])
        assert final_features.shape[1] == 8964, (
            f"Dimension mismatch: Expected 8964, got {final_features.shape[1]}"
        )

        preds = self.xgb_model.predict(final_features)
        return np.array(preds).flatten()

In [ ]:
%%writefile analysis_export.py
"""Standardized artifact export: Case Study bundling + schema assembly (Step 5, issue #17).

This is the pipeline's final assembly step (see plan_realize.md section 4 /
issue #9's Implementation Decisions): it does not compute new explainability
signal itself (that's Steps 2-4, issues #11-#16) - it selects the 15
representative Case Study samples (`case_study_selection.py`), bundles each
with its per-sample detail, and assembles the standard two-artifact export:

- `model_analysis_arrays.npz`: every large multi-dimensional tensor
  (ISM/mismatch/attention/SHAP arrays for the full Testset).
- `model_analysis_summary.json`: metadata, `global_evaluation`,
  `complex_mismatch_metadata`, and the `case_studies` list - no large arrays,
  per issue #9's AC24/AC25.
"""

import json

import numpy as np

# Order matches Model_B_Predictor.compute_physical_features's hstack order
# (mfe, dg, tm, gc) - see model_b_wrapper.py / shap_analysis.py.
PHYSICAL_FEATURE_KEYS = ("mfe", "dg", "tm", "gc")

PHYSICAL_SHAP_POSITIVE_INTERPRETATION = (
    "Physical features' SHAP contribution shifted Model B's prediction "
    "toward the ground truth."
)
PHYSICAL_SHAP_NEGATIVE_INTERPRETATION = (
    "Physical features' SHAP contribution shifted Model B's prediction "
    "away from the ground truth."
)

REQUIRED_TOP_LEVEL_KEYS = (
    "metadata",
    "global_evaluation",
    "complex_mismatch_metadata",
    "case_studies",
)

MAX_CASE_STUDIES = 15

# Large multi-dimensional arrays that must live only in the .npz - never
# embedded as a JSON key anywhere in the summary (issue #9 AC24/#17 AC5).
LARGE_ARRAY_KEYS = frozenset(
    {
        "ism_delta",
        "ism_delta_model_a",
        "ism_delta_model_b",
        "mismatch_single_delta",
        "mismatch_single_delta_model_a",
        "mismatch_single_delta_model_b",
        "mismatch_single_min",
        "mismatch_single_min_model_a",
        "mismatch_single_min_model_b",
        "mismatch_complex",
        "mismatch_complex_model_a",
        "mismatch_complex_model_b",
        "attention_rollout",
        "shap_values",
    }
)


def to_jsonable(value):
    """Recursively convert numpy scalars/arrays to plain Python/JSON-safe types."""
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {k: to_jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(v) for v in value]
    return value


def physical_shap_interpretation(
    true_score: float, pred_b: float, physical_shap_values
) -> str:
    """Direction-aware interpretation of a sample's physical-feature SHAP contribution.

    Rather than a fixed narrative string, this compares Model B's actual
    error against the counterfactual error it would have had without the
    physical features' contribution (`pred_b` minus the sum of their SHAP
    values, since SHAP contributions are additive around the model's
    expected value): if removing the physical contribution would have
    pushed the prediction further from `true_score`, the physical features
    are reported as having moved the prediction *toward* the truth, and
    vice versa.
    """
    physical_shap_sum = float(np.sum(physical_shap_values))
    pred_without_physical = pred_b - physical_shap_sum
    error_with_physical = abs(true_score - pred_b)
    error_without_physical = abs(true_score - pred_without_physical)

    if error_with_physical <= error_without_physical:
        return PHYSICAL_SHAP_POSITIVE_INTERPRETATION
    return PHYSICAL_SHAP_NEGATIVE_INTERPRETATION


def build_case_study_entry(
    case: dict,
    *,
    sample_id: int,
    original_id: int,
    sequence: str,
    true_score: float,
    pred_a: float,
    error_a: float,
    pred_b: float,
    error_b: float,
    physical_values: dict,
    physical_shap: dict,
    ig_model_a_norm_attr_36bp,
    ig_model_b_norm_attr_36bp,
    attention_rollout_info: dict,
) -> dict:
    """Bundle one selected Case Study's full per-sample explainability detail.

    `case` is one entry from `case_study_selection.select_case_studies`
    (`case_id`/`case_type`); everything else is that same sample's already-
    computed data. `physical_values`/`physical_shap` are dicts keyed by
    `PHYSICAL_FEATURE_KEYS`. Large arrays (the attention matrix itself) are
    deliberately not embedded here - only `attention_rollout_info`'s
    dimensions/note - per issue #9 AC24/#17 AC5; the IG attribution arrays
    are a fixed, small 36-length vector so they're embedded directly, same
    as plan_realize.md section 4's schema example.
    """
    interpretation = physical_shap_interpretation(
        true_score, pred_b, [physical_shap[k] for k in PHYSICAL_FEATURE_KEYS]
    )

    return {
        "case_id": case["case_id"],
        "sample_id": int(sample_id),
        "original_id": int(original_id),
        "sequence": sequence,
        "true_score": float(true_score),
        "model_a": {"pred_raw": float(pred_a), "error": float(error_a)},
        "model_b": {"pred_raw": float(pred_b), "error": float(error_b)},
        "case_type": case["case_type"],
        "per_sample_physical_values": {
            k: float(physical_values[k]) for k in PHYSICAL_FEATURE_KEYS
        },
        "per_sample_physical_shap": {
            **{k: float(physical_shap[k]) for k in PHYSICAL_FEATURE_KEYS},
            "interpretation": interpretation,
        },
        "integrated_gradients": {
            "model_a_norm_attr_36bp": to_jsonable(
                np.asarray(ig_model_a_norm_attr_36bp)
            ),
            "model_b_phase4_projected_norm_attr_36bp": to_jsonable(
                np.asarray(ig_model_b_norm_attr_36bp)
            ),
        },
        "attention_rollout": dict(attention_rollout_info),
    }


def assemble_case_studies(
    selection: list[dict],
    *,
    meta_df,
    pred_a: np.ndarray,
    error_a: np.ndarray,
    pred_b: np.ndarray,
    error_b: np.ndarray,
    physical_values_by_index: dict,
    physical_shap_by_index: dict,
    ig_by_index: dict,
    attention_rollout_info: dict,
) -> list[dict]:
    """Build the full `case_studies` list from a `select_case_studies` selection.

    `pred_a`/`error_a`/`pred_b`/`error_b` are full-Testset-length arrays
    (residual/quantile computation over the whole Testset is what selection
    itself needed), indexed by each case's `index`. `physical_values_by_index`
    /`physical_shap_by_index`/`ig_by_index` instead map a selected sample's
    index directly to its already-computed data (a 4-value
    `PHYSICAL_FEATURE_KEYS`-ordered array-like, and an
    `{"model_a_norm_attr_36bp": ..., "model_b_phase4_projected_norm_attr_36bp": ...}`
    dict, respectively) - physical values/SHAP/Integrated Gradients are only
    ever computed for the (at most 15) selected Case Study samples, never
    the full Testset, since that's all this export needs (issue #17's AC3
    scopes IG/attention detail to the Case Studies themselves), so there is
    no full-population array to index into for these three.
    """
    entries = []
    for case in selection:
        idx = case["index"]
        row = meta_df.iloc[idx]
        entries.append(
            build_case_study_entry(
                case,
                sample_id=row["sample_id"],
                original_id=row["original_id"],
                sequence=row["sequence"],
                true_score=row["true_score"],
                pred_a=pred_a[idx],
                error_a=error_a[idx],
                pred_b=pred_b[idx],
                error_b=error_b[idx],
                physical_values=dict(
                    zip(PHYSICAL_FEATURE_KEYS, physical_values_by_index[idx])
                ),
                physical_shap=dict(
                    zip(PHYSICAL_FEATURE_KEYS, physical_shap_by_index[idx])
                ),
                ig_model_a_norm_attr_36bp=ig_by_index[idx]["model_a_norm_attr_36bp"],
                ig_model_b_norm_attr_36bp=ig_by_index[idx][
                    "model_b_phase4_projected_norm_attr_36bp"
                ],
                attention_rollout_info=attention_rollout_info,
            )
        )
    return entries


def build_model_analysis_summary(
    metadata: dict,
    global_evaluation: dict,
    complex_mismatch_metadata: list,
    case_studies: list,
) -> dict:
    """Assemble the standard `model_analysis_summary.json` top-level dict."""
    return {
        "metadata": metadata,
        "global_evaluation": global_evaluation,
        "complex_mismatch_metadata": complex_mismatch_metadata,
        "case_studies": case_studies,
    }


def _is_leaked_array_value(value) -> bool:
    """Whether `value` looks like actual leaked tensor data, not descriptive metadata.

    An `ndarray` is always a leak. A `list`/`tuple` is only a leak if it
    isn't purely made of strings - real array data (or its post-
    `to_jsonable` nested-list form) is numbers/further nesting, whereas
    `metadata.array_axis_definitions` legitimately holds plain lists of
    human-readable axis-label strings under these same key names (per
    plan_realize.md section 4's schema) and must not be flagged.
    """
    if isinstance(value, np.ndarray):
        return True
    if isinstance(value, (list, tuple)):
        return not all(isinstance(item, str) for item in value)
    return False


def _find_large_array_keys(obj) -> set:
    """Recursively collect any `LARGE_ARRAY_KEYS` name whose *array-shaped* value appears anywhere in `obj`.

    A `LARGE_ARRAY_KEYS` name used as a key is only a violation when its
    value actually looks like leaked array data (see
    `_is_leaked_array_value`). It is legitimate for a Case Study entry's
    small `attention_rollout` *metadata* dict (dims + note, no matrix data -
    see `build_case_study_entry`) and for `metadata.array_axis_definitions`'s
    axis-label-string lists to reuse these same names, so neither is itself
    a violation.
    """
    found = set()
    if isinstance(obj, dict):
        for key, value in obj.items():
            if key in LARGE_ARRAY_KEYS and _is_leaked_array_value(value):
                found.add(key)
            found |= _find_large_array_keys(value)
    elif isinstance(obj, (list, tuple)):
        for item in obj:
            found |= _find_large_array_keys(item)
    return found


def validate_summary_schema(summary: dict) -> None:
    """Validate `summary` against the Section-4 export schema (issue #17 AC4/AC5).

    Checks: all required top-level keys present, `case_studies` count capped
    at `MAX_CASE_STUDIES`, no `LARGE_ARRAY_KEYS` name leaked into the JSON
    (they must live only in the `.npz`), and the summary is actually
    JSON-serializable (numpy scalars/arrays would raise `TypeError` here
    instead of silently producing malformed JSON).
    """
    missing = [key for key in REQUIRED_TOP_LEVEL_KEYS if key not in summary]
    if missing:
        raise ValueError(
            f"model_analysis_summary.json missing required keys: {missing}"
        )

    if len(summary["case_studies"]) > MAX_CASE_STUDIES:
        raise ValueError(
            f"case_studies has {len(summary['case_studies'])} entries, "
            f"exceeding the cap of {MAX_CASE_STUDIES}"
        )

    leaked = _find_large_array_keys(summary)
    if leaked:
        raise ValueError(
            f"Large array key(s) {sorted(leaked)} must not appear in "
            "model_analysis_summary.json - they belong only in model_analysis_arrays.npz"
        )

    json.dumps(summary)  # raises TypeError on any un-jsonified numpy leakage


def export_artifacts(
    summary: dict,
    arrays: dict,
    json_path: str,
    npz_path: str,
) -> None:
    """Validate and write the two standard export artifacts.

    `arrays` is everything destined for `model_analysis_arrays.npz` (full
    Testset ISM/mismatch/attention/SHAP tensors); `summary` is the
    JSON-only metadata + Case Studies dict. Validation runs before either
    file is written, so a schema violation never leaves a half-written pair
    on disk.
    """
    validate_summary_schema(summary)

    jsonable_summary = to_jsonable(summary)
    with open(json_path, "w") as f:
        json.dump(jsonable_summary, f, indent=2)

    np.savez(npz_path, **arrays)

In [ ]:
%%writefile model_b_testset_export.py
"""Model B predictions + physical features for the full Testset (issue #19).

`model_analysis_summary.json` only carries Model B's prediction and its 4
physical features (MFE, ΔG, Tm, GC) for the 15 Case Studies. The Model B
explanation dashboard (`visualization/plan_draft.md`) needs them for all 514
Testset samples, so this exports one small table keyed by `sample_id`.

It is a CSV rather than another `.npz` key: one row per sample with 6 scalar
columns is a table, not a multi-dimensional tensor, and the dashboard reads
it straight into Pandas.

Integrated Gradients stays limited to the Case Studies (out of scope here).
"""

import numpy as np
import pandas as pd
from analysis_export import PHYSICAL_FEATURE_KEYS

EXPORT_COLUMNS = ("sample_id", "pred_raw", *PHYSICAL_FEATURE_KEYS)

# Case Study values were produced by an earlier Colab GPU run; a fresh NT
# forward pass can differ by float32/cuDNN noise, never by a real amount.
CASE_STUDY_MATCH_ATOL = 1e-4


def build_model_b_testset_table(
    meta_df: pd.DataFrame, pred_b: np.ndarray, physical: np.ndarray
) -> pd.DataFrame:
    """One row per Testset sample: `sample_id`, `pred_raw`, then `mfe`/`dg`/`tm`/`gc`.

    `physical` is `Model_B_Predictor.compute_physical_features`'s `(n, 4)`
    output, in `PHYSICAL_FEATURE_KEYS` order.
    """
    n = len(meta_df)
    pred_b = np.asarray(pred_b, dtype=np.float64).reshape(-1)
    physical = np.asarray(physical, dtype=np.float64)

    if len(pred_b) != n:
        raise ValueError(f"pred_b has {len(pred_b)} rows, Testset has {n}")
    if physical.shape != (n, len(PHYSICAL_FEATURE_KEYS)):
        raise ValueError(
            f"physical has shape {physical.shape}, expected "
            f"({n}, {len(PHYSICAL_FEATURE_KEYS)})"
        )
    sample_ids = meta_df["sample_id"].to_numpy(dtype=np.int64)
    if not np.array_equal(sample_ids, np.arange(n)):
        raise ValueError(f"sample_id must be exactly 0..{n - 1} in row order")
    if np.isnan(pred_b).any() or np.isnan(physical).any():
        raise ValueError("pred_b/physical contain NaN")

    table = pd.DataFrame({"sample_id": sample_ids, "pred_raw": pred_b})
    for j, key in enumerate(PHYSICAL_FEATURE_KEYS):
        table[key] = physical[:, j]
    return table


def check_case_studies_match(
    table: pd.DataFrame, case_studies: list[dict], atol: float = CASE_STUDY_MATCH_ATOL
) -> None:
    """Raise `ValueError` listing every Case Study field that disagrees with `table`.

    `case_studies` is `model_analysis_summary.json`'s `case_studies` list.
    """
    rows = table.set_index("sample_id")
    mismatches = []
    for case in case_studies:
        sample_id = case["sample_id"]
        if sample_id not in rows.index:
            mismatches.append(f"sample_id={sample_id}: not in table")
            continue
        row = rows.loc[sample_id]
        expected = {
            "pred_raw": case["model_b"]["pred_raw"],
            **case["per_sample_physical_values"],
        }
        for key, value in expected.items():
            if not np.isclose(row[key], value, rtol=0.0, atol=atol):
                mismatches.append(
                    f"sample_id={sample_id} {key}: table={row[key]!r}, "
                    f"case study={value!r}"
                )
    if mismatches:
        raise ValueError(
            "Model B Testset export disagrees with the Case Studies:\n"
            + "\n".join(mismatches)
        )


def write_model_b_testset_table(table: pd.DataFrame, path) -> None:
    table.to_csv(path, index=False, columns=list(EXPORT_COLUMNS))


def load_model_b_testset_table(path) -> pd.DataFrame:
    return pd.read_csv(path, dtype={"sample_id": np.int64})[list(EXPORT_COLUMNS)]

In [ ]:
%%writefile export_model_b_testset.py
"""Runner: Model B predictions + physical features for all 514 Testset samples (issue #19).

Run on Colab GPU (`colab_model_b_testset_export.ipynb`) - the NT forward pass
over the whole Testset does not fit this machine's RAM. Requires (in the
working directory):

- `test_metadata.csv` (issue #1, `test_dataset_verifier.py`)
- `NT_sacas9_fintuned_model/` + `hybrid_xgb_model.json` (Model B)
- `final_analysis_result/model_analysis_summary.json` (issue #17, `step5_export.py`)

Writes `final_analysis_result/model_b_testset_predictions.csv` only after its
15 Case Study rows are confirmed to match the values already in
`model_analysis_summary.json`.
"""

import json
import os

import pandas as pd
from model_b_testset_export import (
    build_model_b_testset_table,
    check_case_studies_match,
    write_model_b_testset_table,
)
from model_b_wrapper import Model_B_Predictor

TEST_METADATA_PATH = "test_metadata.csv"
NT_MODEL_DIR = "./NT_sacas9_fintuned_model"
XGB_MODEL_PATH = "hybrid_xgb_model.json"
SUMMARY_PATH = os.path.join("final_analysis_result", "model_analysis_summary.json")
OUTPUT_PATH = os.path.join("final_analysis_result", "model_b_testset_predictions.csv")


def _require(path: str, produced_by: str) -> str:
    # Same phrasing as every other stepN_*.py runner in this repo.
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{path} not found. {produced_by}를 먼저 실행해 주세요."
        )
    return path


def main():
    print("=== Model B Testset Export (predictions + physical features) ===")

    meta_df = pd.read_csv(_require(TEST_METADATA_PATH, "test_dataset_verifier.py"))
    with open(_require(SUMMARY_PATH, "step5_export.py"), encoding="utf-8") as f:
        case_studies = json.load(f)["case_studies"]
    sequences = meta_df["sequence"].tolist()
    print(f"Loaded testset: {len(sequences)} samples.")

    predictor_b = Model_B_Predictor(
        nt_model_dir=NT_MODEL_DIR, xgb_model_path=XGB_MODEL_PATH
    )

    print("\nScoring the Testset with Model B...")
    pred_b = predictor_b.predict(sequences)
    print("Computing physical features (MFE, dG, Tm, GC)...")
    physical = predictor_b.compute_physical_features(sequences)

    table = build_model_b_testset_table(meta_df, pred_b, physical)

    print(
        f"\nChecking the {len(case_studies)} Case Study rows against {SUMMARY_PATH}..."
    )
    check_case_studies_match(table, case_studies)
    print(" - All Case Study rows match.")

    print(f"\nWriting {OUTPUT_PATH}...")
    write_model_b_testset_table(table, OUTPUT_PATH)
    print("Done.")


if __name__ == "__main__":
    main()

## 5. Run the export

In [ ]:
!python export_model_b_testset.py

## 6. Sanity check

Download `final_analysis_result/model_b_testset_predictions.csv` from Drive
into your local `xai_test/final_analysis_result/` and commit it.

In [ ]:
from model_b_testset_export import load_model_b_testset_table

table = load_model_b_testset_table(
    "final_analysis_result/model_b_testset_predictions.csv"
)
assert len(table) == 514 and table["sample_id"].tolist() == list(range(514))
assert not table.isna().any().any()
print(table.head())
print(table.describe())